In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
file_path = r"C:\Users\tejas\Downloads\saas_project\Telco_customer_churn.xlsx"

df = pd.read_excel(file_path)

In [3]:
# Convert Total Charges to numeric
df['Total Charges'] = pd.to_numeric(
    df['Total Charges'],
    errors='coerce'
)

# Fill missing values
df['Total Charges'] = df['Total Charges'].fillna(0)

In [4]:
# Rename columns to SaaS-style naming

df = df.rename(columns={
    'CustomerID': 'customer_id',
    'Tenure Months': 'tenure_months',
    'Monthly Charges': 'monthly_price',
    'Total Charges': 'total_revenue',
    'Contract': 'subscription_type',
    'Payment Method': 'payment_method',
    'Churn Value': 'churn'
})

In [5]:
saas_df = df[
    [
        'customer_id',
        'tenure_months',
        'subscription_type',
        'monthly_price',
        'total_revenue',
        'payment_method',
        'Internet Service',
        'churn'
    ]
].copy()

In [6]:
saas_df = saas_df.rename(
    columns={
        'Internet Service': 'plan_type'
    }
)

saas_df.columns = saas_df.columns.str.lower()

In [7]:
mrr = saas_df['monthly_price'].sum()

arr = mrr * 12

churn_rate = saas_df['churn'].mean()

active_customers = saas_df[
    saas_df['churn'] == 0
].shape[0]

churned_customers = saas_df[
    saas_df['churn'] == 1
].shape[0]

print("Total MRR:", round(mrr, 2))
print("Total ARR:", round(arr, 2))
print("Churn Rate:", round(churn_rate * 100, 2), "%")
print("Active Customers:", active_customers)
print("Churned Customers:", churned_customers)

Total MRR: 456116.6
Total ARR: 5473399.2
Churn Rate: 26.54 %
Active Customers: 5174
Churned Customers: 1869


In [8]:
revenue_by_subscription = (
    saas_df
    .groupby('subscription_type')['monthly_price']
    .sum()
)

revenue_by_plan = (
    saas_df
    .groupby('plan_type')['monthly_price']
    .sum()
)

print(revenue_by_subscription)
print(revenue_by_plan)

subscription_type
Month-to-month    257294.15
One year           95816.60
Two year          103005.85
Name: monthly_price, dtype: float64
plan_type
DSL            140665.35
Fiber optic    283284.40
No              32166.85
Name: monthly_price, dtype: float64


In [9]:
arpu = saas_df['monthly_price'].mean()

active_arpu = saas_df[
    saas_df['churn'] == 0
]['monthly_price'].mean()

avg_lifetime = 1 / churn_rate

ltv = arpu * avg_lifetime

cac = 120

ltv_cac_ratio = ltv / cac

payback_period = cac / arpu

print("ARPU:", round(arpu, 2))
print("Active ARPU:", round(active_arpu, 2))
print("LTV:", round(ltv, 2))
print("LTV:CAC:", round(ltv_cac_ratio, 2))
print("Payback:", round(payback_period, 2))

ARPU: 64.76
Active ARPU: 61.27
LTV: 244.04
LTV:CAC: 2.03
Payback: 1.85


In [10]:
churn_by_sub = (
    saas_df
    .groupby('subscription_type')['churn']
    .mean()
)

churn_by_plan = (
    saas_df
    .groupby('plan_type')['churn']
    .mean()
)

print(round(churn_by_sub * 100, 2))
print(round(churn_by_plan * 100, 2))

subscription_type
Month-to-month    42.71
One year          11.27
Two year           2.83
Name: churn, dtype: float64
plan_type
DSL            18.96
Fiber optic    41.89
No              7.40
Name: churn, dtype: float64


In [11]:
df_model = df.copy()

df_model['risk_score'] = 0

df_model.loc[
    df_model['subscription_type'] == 'Month-to-month',
    'risk_score'
] += 2

df_model.loc[
    df_model['Internet Service'] == 'Fiber optic',
    'risk_score'
] += 1

df_model.loc[
    df_model['tenure_months'] < 12,
    'risk_score'
] += 2

In [12]:
df_ai = df[
    [
        'tenure_months',
        'subscription_type',
        'monthly_price',
        'total_revenue',
        'payment_method',
        'Internet Service',
        'churn'
    ]
].copy()

df_ai['total_revenue'] = pd.to_numeric(
    df_ai['total_revenue'],
    errors='coerce'
)

df_ai = df_ai.dropna()

df_ai = pd.get_dummies(
    df_ai,
    drop_first=True
)

In [13]:
X = df_ai.drop('churn', axis=1)

y = df_ai['churn']

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [15]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=2000)

model.fit(
    X_train_scaled,
    y_train
)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,2000
,multi_class,'deprecated'


In [16]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

y_pred = model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

Accuracy: 0.7913413768630234
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1009
           1       0.66      0.54      0.59       400

    accuracy                           0.79      1409
   macro avg       0.75      0.71      0.73      1409
weighted avg       0.78      0.79      0.78      1409



In [17]:
y_pred_prob = model.predict_proba(
    X_test_scaled
)[:, 1]

results = X_test.copy()

results['actual_churn'] = y_test

results['churn_probability'] = y_pred_prob

results['monthly_price'] = df.loc[
    X_test.index,
    'monthly_price'
]

results['high_risk_flag'] = (
    results['churn_probability'] > 0.6
)

revenue_at_risk = results[
    results['high_risk_flag']
]['monthly_price'].sum()

revenue_saved = revenue_at_risk * 0.30

print("Revenue at Risk:", revenue_at_risk)

print("Revenue Saved:", revenue_saved)

Revenue at Risk: 17437.15
Revenue Saved: 5231.145


In [18]:
importance_df = pd.DataFrame({
    'feature': X.columns,
    'coefficient': model.coef_[0]
})

importance_df = importance_df.sort_values(
    by='coefficient',
    ascending=False
)

importance_df

,feature,coefficient
2,total_revenue,0.786399
8,Internet Service_Fiber optic,0.451715
6,payment_method_Electronic check,0.202532
5,payment_method_Credit card (automatic),-0.041727
7,payment_method_Mailed check,-0.047211
1,monthly_price,-0.063192
3,subscription_type_One year,-0.307616
9,Internet Service_No,-0.392133
4,subscription_type_Two year,-0.738202
0,tenure_months,-1.506592


In [19]:
X_scaled_full = scaler.transform(X)

all_probabilities = model.predict_proba(
    X_scaled_full
)[:, 1]

df['predicted_churn_probability'] = (
    all_probabilities
)

In [20]:
y_pred = model.predict(X_scaled_full)

accuracy = accuracy_score(y, y_pred)

precision = precision_score(y, y_pred)

recall = recall_score(y, y_pred)

f1 = f1_score(y, y_pred)

cm = confusion_matrix(y, y_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y, y_pred))

Accuracy: 0.7978134317762318
Precision: 0.644199611147116
Recall: 0.5318352059925093
F1 Score: 0.5826494724501758

Confusion Matrix:
[[4625  549]
 [ 875  994]]

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.89      0.87      5174
           1       0.64      0.53      0.58      1869

    accuracy                           0.80      7043
   macro avg       0.74      0.71      0.72      7043
weighted avg       0.79      0.80      0.79      7043



In [21]:
df.to_csv(
    r"C:\Users\tejas\Downloads\saas_project\saas_ai_dashboard_ready.csv",
    index=False
)

print("Dashboard dataset exported successfully.")

Dashboard dataset exported successfully.
